# Week 26, Day 1: Review I: arrays, vectors, matrices and least squares

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w026/d1.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

### Exercise: Problem 1: decoding direction from a population

100 neurons have cosine tuning, rate = 10 + 8 cos(θ − θ_pref), with Poisson noise. Simulate 200 trials at random directions (local rng seed 4) into `rates`. Write `population_vector(rates)` without loops: subtract each neuron's baseline of 10, weight each neuron's preferred unit vector by the result, sum, and return the angle of the sum. Store the decoded angles in `decoded`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
r4 = np.random.default_rng(4)
pref = r4.uniform(0, 2 * np.pi, 100)                 # preferred directions
theta = r4.uniform(0, 2 * np.pi, 200)                # true direction on each trial
mean = 10 + 8 * np.cos(theta[:, None] - pref[None, :])   # broadcasting: trials x neurons
rates = r4.poisson(mean).astype(float)

def population_vector(rates):
    w = rates - 10                                    # trials x neurons
    x = ...                                           # sum over neurons of w cos(pref)
    y = ...
    return np.arctan2(y, x)

decoded = population_vector(rates)
err = np.angle(np.exp(1j * (decoded - theta)))
print(f"median absolute error {np.degrees(np.median(np.abs(err))):.1f} degrees")

In [ ]:
import numpy as np
def test_rates():
    "rates has one row per trial and one column per neuron, all non-negative"
    assert rates.shape == (200, 100) and np.all(rates >= 0)

def test_decode():
    "decoded holds an angle per trial, and the median absolute error is below 0.25 rad"
    err = np.angle(np.exp(1j * (decoded - theta)))
    assert decoded.shape == (200,) and np.median(np.abs(err)) < 0.25

def test_no_loop():
    "the decoder uses no Python loop"
    import dis
    ops = {i.opname for i in dis.get_instructions(population_vector)}
    assert "FOR_ITER" not in ops

test_rates()
test_decode()
test_no_loop()
print("All checks passed")

### Exercise: Problem 2: a regularised encoding model

A neuron's response is a weighted sum of 60 stimulus features plus noise, with only 80 training trials. Fit ordinary least squares `w_ols` by solving the normal equations, write `ridge_fit(lam)`, and find the held-out R² of ridge over the given lambdas; store the best in `best_r2`.

In [ ]:
r5 = np.random.default_rng(5)
p = 60
w_true = r5.normal(size=p) * (r5.random(p) < 0.3)   # a few features matter
Xtr, Xte = r5.normal(size=(80, p)), r5.normal(size=(400, p))
ytr = Xtr @ w_true + 2.0 * r5.normal(size=80)
yte = Xte @ w_true + 2.0 * r5.normal(size=400)

def r2_test(w):
    return 1 - np.sum((yte - Xte @ w) ** 2) / np.sum((yte - yte.mean()) ** 2)

w_ols = ...
def ridge_fit(lam):
    return ...

lams = [0.1, 1, 3, 10, 30, 100, 300]
scores = [r2_test(ridge_fit(l)) for l in lams]
best_r2 = max(scores)
print(f"least squares held-out R2 {r2_test(w_ols):.3f};  ridge:", {l: round(float(v), 3) for l, v in zip(lams, scores)})

In [ ]:
import numpy as np
def test_ols():
    "w_ols solves the normal equations on the training data"
    assert np.allclose(Xtr.T @ Xtr @ w_ols, Xtr.T @ ytr)

def test_ridge():
    "ridge_fit(lam) solves (X^T X + lam I) w = X^T y"
    w = ridge_fit(10.0)
    assert np.allclose((Xtr.T @ Xtr + 10.0 * np.eye(Xtr.shape[1])) @ w, Xtr.T @ ytr)

def test_better():
    "the best ridge model beats least squares on held-out data"
    assert best_r2 > r2_test(w_ols) + 0.05

test_ols()
test_ridge()
test_better()
print("All checks passed")

### Exercise: Problem 3: an orthonormal basis for a population subspace

Three patterns of activity over 20 neurons are the columns of A. Write `gram_schmidt(A)` (subtract from each column its projections onto the previous orthonormal vectors, then normalise) to get Q, then project the population state x onto their span to get `proj`, and report how much of x's squared length lies in the subspace.

In [ ]:
r6 = np.random.default_rng(6)
A = r6.normal(size=(20, 3)); x = r6.normal(size=20)

def gram_schmidt(A):
    Q = np.zeros_like(A)
    for j in range(A.shape[1]):
        v = A[:, j].copy()
        for i in range(j):
            v = ...                                   # remove the component along Q[:, i]
        Q[:, j] = v / np.linalg.norm(v)
    return Q

Q = gram_schmidt(A)
proj = ...
print(f"{100 * np.sum(proj ** 2) / np.sum(x ** 2):.0f}% of x's squared length lies in the 3-dimensional subspace (3/20 = 15% expected at random)")

In [ ]:
import numpy as np
def test_ortho():
    "Q has orthonormal columns"
    assert np.allclose(Q.T @ Q, np.eye(3))

def test_span():
    "Q spans the same subspace as the columns of A"
    assert np.allclose(Q @ (Q.T @ A), A)

def test_proj():
    "proj is the projection of x onto the subspace, and the residual is orthogonal to it"
    assert np.allclose(proj, Q @ (Q.T @ x)) and np.allclose(Q.T @ (x - proj), 0)

test_ortho()
test_span()
test_proj()
print("All checks passed")